# Genealogy-over-surface — full pipeline

Clones the repo, extracts XLM-R-large representations for all 25 languages, then runs Tier 0 / 1 / 2 analyses and zips `/kaggle/working/results`.

Expected total runtime on a T4: extraction ~30-45 min, analyses ~15 min. Save outputs at the end.

In [ ]:
# 0. clone / pull the repo
import os, subprocess, sys
REPO_URL = 'https://github.com/saketatreya/rep-phylogeny.git'
WORK = '/kaggle/working/rep-phylogeny'
if not os.path.exists(WORK):
    subprocess.run(['git', 'clone', REPO_URL, WORK], check=True)
else:
    subprocess.run(['git', '-C', WORK, 'pull', '--ff-only'], check=True)
os.chdir(WORK)
print('cwd =', os.getcwd())
subprocess.run(['git', 'log', '--oneline', '-1'], check=True)

In [ ]:
# 1. install deps (Kaggle has most of these; add pinned versions for safety)
!pip install -q -r requirements.txt

In [ ]:
# 2. extract XLM-R-large representations (GPU)
!python run_extract.py --out-dir /kaggle/working/results/reps

In [ ]:
# 3. Tier 0 — atomic flip {English, German, French}
!python run_tier0.py --reps-dir /kaggle/working/results/reps --out-dir /kaggle/working/results

In [ ]:
# 4. Tier 1 — held-out family discriminant
!python run_tier1.py --reps-dir /kaggle/working/results/reps --out-dir /kaggle/working/results

In [ ]:
# 5. Tier 2 — NJ trees, axis transfer, Mantel, transliteration
!python run_tier2.py --reps-dir /kaggle/working/results/reps --out-dir /kaggle/working/results

In [ ]:
# 6. SUMMARY
!cat /kaggle/working/results/SUMMARY.txt

In [ ]:
# 7. zip results for download (exclude the heavy reps/ subtree by default; pass
# --reps to include them too if you want to rerun analyses locally).
import sys
INCLUDE_REPS = False
import shutil
if INCLUDE_REPS:
    shutil.make_archive('/kaggle/working/results_full', 'zip', '/kaggle/working/results')
else:
    import os, zipfile
    with zipfile.ZipFile('/kaggle/working/results_analyses.zip', 'w', zipfile.ZIP_DEFLATED) as z:
        for root, _, files in os.walk('/kaggle/working/results'):
            if '/reps' in root:
                continue
            for f in files:
                p = os.path.join(root, f)
                z.write(p, os.path.relpath(p, '/kaggle/working/results'))
print('done')